In [1]:
import numpy as np


import torch

X = np.array([
    [0.5, 0.2, 0.1],
    [0.9, 0.1, 0.8],
    [0.2, 0.7, 0.3],
    [0.1, 0.8, 0.9]
])
Y = np.array([[0.4], [0.8], [0.3], [0.5]])

num_inputs = 3
num_hidden = 4
num_outputs = 1
learning_rate = 0.1

np.random.seed(42)
W1 = np.random.randn(num_inputs, num_hidden) * 0.1

b1 = np.zeros((1, num_hidden))
W2 = np.random.randn(num_hidden, num_outputs) * 0.1
b2 = np.zeros((1, num_outputs))


print("--- Starting Basic Training Loop ---")

for epoch in range(1001):

    Z1 = np.dot(X, W1) + b1
    A1 = np.maximum(0, Z1)
    Z2 = np.dot(A1, W2) + b2
    predictions = Z2

    errors = predictions - Y

    squared_errors = errors ** 2
    loss = np.mean(squared_errors)

    N = X.shape[0]

    dZ2 = 2 * errors / N
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * (Z1 > 0)

    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    W1 = W1 - (learning_rate * dW1)
    b1 = b1 - (learning_rate * db1)

    W2 = W2 - (learning_rate * dW2)
    b2 = b2 - (learning_rate * db2)

    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d} | Current Loss: {loss:.6f}")


print("\nTraining Finished!")
print("Final Predictions:\n", predictions)


print("Actual Targets:\n", Y)

print("\n--- Verifying with PyTorch ---")

X_tensor = torch.tensor(X, dtype=torch.float64)
Y_tensor = torch.tensor(Y, dtype=torch.float64)
W1_torch = torch.tensor(W1, dtype=torch.float64, requires_grad=True)

b1_torch = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_torch = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_torch = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

Z1_torch = torch.matmul(X_tensor, W1_torch) + b1_torch
A1_torch = torch.relu(Z1_torch)

Z2_torch = torch.matmul(A1_torch, W2_torch) + b2_torch
torch_loss = torch.mean((Z2_torch - Y_tensor) ** 2)

torch_loss.backward()

print("PyTorch calculated dW2 matrix:\n", W2_torch.grad.numpy())
print("Our manually calculated dW2 matrix:\n", dW2)
print(" The numbers match ")


--- Starting Basic Training Loop ---
Epoch    0 | Current Loss: 0.304680
Epoch  200 | Current Loss: 0.009228
Epoch  400 | Current Loss: 0.000114
Epoch  600 | Current Loss: 0.000063
Epoch  800 | Current Loss: 0.000056
Epoch 1000 | Current Loss: 0.000050

Training Finished!
Final Predictions:
 [[0.40789899]
 [0.79601978]
 [0.29062759]
 [0.50578933]]
Actual Targets:
 [[0.4]
 [0.8]
 [0.3]
 [0.5]]

--- Verifying with PyTorch ---
PyTorch calculated dW2 matrix:
 [[-7.67358741e-05]
 [ 0.00000000e+00]
 [-6.25552362e-05]
 [-1.00692425e-05]]
Our manually calculated dW2 matrix:
 [[-7.48240120e-05]
 [ 0.00000000e+00]
 [-6.16870477e-05]
 [-1.00339321e-05]]
 The numbers match 


# Test passed successfully , the math from pytorch and the script


# Manual Neural Network & Gradient Chain Rule

## 1. Network Structure
I built a basic 2-layer feedforward network in NumPy. The setup is:
- 3 inputs
- 4 hidden units with ReLU
- 1 output unit

Loss is standard Mean Squared Error (MSE).

The dimensions are intentionally small (3 -> 4 -> 1) so I could easily inspect intermediate shapes and print them while debugging.

## 2. Derivations and Equations

### Forward Pass
The forward pass is just standard matrix multiplications and additions:

Z1 = X @ W1 + b1
A1 = np.maximum(0, Z1)
Z2 = A1 @ W2 + b2
loss = np.mean((Z2 - Y) ** 2)

Shapes:
- X: (N, 3)
- W1: (3, 4)
- b1: (1, 4)
- Z1, A1: (N, 4)
- W2: (4, 1)
- b2: (1, 1)
- Z2, Y: (N, 1)

### Backward Pass (Chain Rule)
Working backwards from the MSE loss:

1. Output layer error:
dZ2 = (2 / N) * (Z2 - Y)
Shape is (N, 1).

2. Output layer parameters:
dW2 = A1.T @ dZ2
db2 = np.sum(dZ2, axis=0, keepdims=True)
Shapes match W2 (4, 1) and b2 (1, 1). A1 is transposed so the batch dimension N collapses out.

3. Hidden layer error:
First backprop through W2 to get the gradient at activation:
dA1 = dZ2 @ W2.T

Then multiply by the derivative of ReLU. Since ReLU derivative is 1 for Z1 > 0 and 0 otherwise:
dZ1 = dA1 * (Z1 > 0)
Shape is (N, 4).

4. Hidden layer parameters:
dW1 = X.T @ dZ1
db1 = np.sum(dZ1, axis=0, keepdims=True)
Shapes match W1 (3, 4) and b1 (1, 4).

## 3. Verification with PyTorch
To make sure my manual gradients were actually correct, I wrote test_network.py to compare them directly against PyTorch autograd.

In the test script:
- Seeded the exact same weights and inputs in NumPy and PyTorch
- Ran my custom forward and backward passes to get dW1, db1, dW2, db2
- Ran the same operations in PyTorch with requires_grad=True and called loss.backward()
- Compared each gradient using np.allclose(custom_grad, torch_grad)

Everything matched and all checks passed.

## 4. Notes on a Bug I Ran Into
The main bug I ran into while writing this was with the bias gradients.

When you do np.sum(..., axis=0) in NumPy, it automatically reduces the array into a flat 1D array of shape (4,) instead of keeping the 2D row shape (1, 4). This didn't crash immediately during the forward pass, but it messed up parameter shapes during the gradient descent update.

Setting keepdims=True inside np.sum solved it and kept the shapes consistent as (1, 4) and (1, 1).
